In [ ]:
import pypsa
import pandas as pd
import numpy as np

from base import load_time_series, load_hydro_data, load_technology_data

In [ ]:
import pypsa
import pandas as pd
import numpy as np

from base import load_time_series, load_hydro_data, load_technology_data

### North 

In [82]:
costs = load_technology_data()
def build_network(costs, ):     
      # Initialize network
      n = pypsa.Network()
      year = 2015
      snapshots = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00", freq="h")
      if len(snapshots) == 8784:
            snapshots = snapshots[~((snapshots.month == 2) & (snapshots.day == 29))]
      n.set_snapshots(snapshots)

      # Add North bus
      n.add("Bus", name="DEU_N_elec", x=9.5, y=53.5, carrier="AC")

      # Load data
      
      df_elec, df_onwind, df_offwind, df_solar = load_time_series()
      p_max_pu_ror = load_hydro_data("DEU", n)

      # Add load
      n.add("Load", "DEU_N_load", bus="DEU_N_elec", p_set=0.4 * df_elec["DEU"].values)

      # Add carriers
      carriers=["onwind","offwind","solar","OCGT","CCGT","hydro","ror","coal","lignite","biomass CHP","battery storage",]
      n.add("Carrier", carriers, co2_emissions=[costs.at[c, "CO2 intensity"] for c in carriers])

      # Capacity factors
      CF_onwind = df_onwind["DEU"][[t.strftime("%Y-%m-%dT%H:%M:%SZ") for t in snapshots]].values
      CF_offwind = df_offwind["DEU"][[t.strftime("%Y-%m-%dT%H:%M:%SZ") for t in snapshots]].values
      CF_solar = df_solar["DEU"][[t.strftime("%Y-%m-%dT%H:%M:%SZ") for t in snapshots]].values
      CF_ror = p_max_pu_ror.loc[n.snapshots].values

      # Add VRE generators (unlimited)
      n.add("Generator", "onwind_N", bus="DEU_N_elec", carrier="onwind", p_nom_extendable=True,
            capital_cost=costs.at["onwind", "capital_cost"], marginal_cost=costs.at["onwind", "marginal_cost"],
            p_max_pu=CF_onwind*1.2, efficiency=costs.at["onwind", "efficiency"], overwrite=True)

      n.add("Generator", "offwind_N", bus="DEU_N_elec", carrier="offwind", p_nom_extendable=True,
            capital_cost=costs.at["offwind", "capital_cost"], marginal_cost=costs.at["offwind", "marginal_cost"],
            p_max_pu=CF_offwind, efficiency=costs.at["offwind", "efficiency"], overwrite=True)

      n.add("Generator", "solar_N", bus="DEU_N_elec", carrier="solar", p_nom_extendable=True,
            capital_cost=costs.at["solar", "capital_cost"], marginal_cost=costs.at["solar", "marginal_cost"],
            p_max_pu=CF_solar*0.8, efficiency=costs.at["solar", "efficiency"], overwrite=True)

      # Add limited generators (with 0.5 factor)
      limits = {
      "coal": 26000 * 0.5,
      "lignite": 21000 * 0.5,
      "biomass CHP": 10000 * 0.5,
      "OCGT": 31000 * 0.5,
      "ror": 6000 * 0.5
      }

      n.add("Generator", "coal_N", bus="DEU_N_elec", carrier="coal", p_nom_extendable=True,
            p_nom_max=limits["coal"], p_min_pu=0.33,
            capital_cost=costs.at["coal", "capital_cost"], marginal_cost=costs.at["coal", "marginal_cost"],
            efficiency=costs.at["coal", "efficiency"], overwrite=True)

      n.add("Generator", "lignite_N", bus="DEU_N_elec", carrier="lignite", p_nom_extendable=True,
            p_nom_max=limits["lignite"], p_min_pu=0.4,
            capital_cost=costs.at["lignite", "capital_cost"], marginal_cost=costs.at["lignite", "marginal_cost"],
            efficiency=costs.at["lignite", "efficiency"], overwrite=True)

      n.add("Generator", "biomass_N", bus="DEU_N_elec", carrier="biomass CHP", p_nom_extendable=True,
            p_nom_max=limits["biomass CHP"], p_min_pu=0.33,
            capital_cost=costs.at["biomass CHP", "capital_cost"], marginal_cost=costs.at["biomass CHP", "marginal_cost"],
            efficiency=costs.at["biomass CHP", "efficiency"], overwrite=True)

      n.add("Generator", "OCGT_N", bus="DEU_N_elec", carrier="OCGT", p_nom_extendable=True,
            p_nom_max=limits["OCGT"], p_min_pu=0.2,
            capital_cost=costs.at["OCGT", "capital_cost"], marginal_cost=costs.at["OCGT", "marginal_cost"],
            efficiency=costs.at["OCGT", "efficiency"], overwrite=True)

      n.add("Generator", "ror_N", bus="DEU_N_elec", carrier="ror", p_nom_extendable=True,
            p_nom_max=limits["ror"], p_max_pu=CF_ror,
            capital_cost=costs.at["ror", "capital_cost"], marginal_cost=costs.at["ror", "marginal_cost"],
            efficiency=costs.at["ror", "efficiency"], overwrite=True)

      # Add large battery
      n.add("Bus", "battery_bus_N", carrier="battery storage")
      n.add("Store", "battery_N", bus="battery_bus_N", e_nom_extendable=True, e_cyclic=True,
            capital_cost=costs.at["battery storage", "capital_cost"])

      n.add("Link", "charge_battery_N", bus0="DEU_N_elec", bus1="battery_bus_N", p_nom_extendable=True,
            efficiency=costs.at["battery inverter", "efficiency"],
            capital_cost=costs.at["battery inverter", "capital_cost"])

      n.add("Link", "discharge_battery_N", bus0="battery_bus_N", bus1="DEU_N_elec", p_nom_extendable=True,
            efficiency=costs.at["battery inverter", "efficiency"],
            capital_cost=costs.at["battery inverter", "capital_cost"])
      
      # Add South bus
      n.add("Bus", name="DEU_S_elec", x=10.5, y=48.5, carrier="AC")

      # Add South load (60% of total demand)
      n.add("Load", "DEU_S_load", bus="DEU_S_elec", p_set=0.6 * df_elec["DEU"].values)

      # Capacity factors
      CF_ror = p_max_pu_ror.loc[n.snapshots].values

      # Add VRE generators (extendable)
      n.add("Generator", "onwind_S", bus="DEU_S_elec", carrier="onwind",p_nom_extendable=True, p_nom_max=20000,
            capital_cost=costs.at["onwind", "capital_cost"], marginal_cost=costs.at["onwind", "marginal_cost"],
            p_max_pu=CF_onwind*0.8, efficiency=costs.at["onwind", "efficiency"], overwrite=True)

      n.add("Generator", "solar_S", bus="DEU_S_elec", carrier="solar", p_nom_extendable=True,
            capital_cost=costs.at["solar", "capital_cost"], marginal_cost=costs.at["solar", "marginal_cost"],
            p_max_pu=CF_solar, efficiency=costs.at["solar", "efficiency"], overwrite=True)

      # Add limited dispatchable generators
      n.add("Generator", "coal_S", bus="DEU_S_elec", carrier="coal", p_nom_extendable=True,
            p_nom_max=limits["coal"], p_min_pu=0.33,
            capital_cost=costs.at["coal", "capital_cost"], marginal_cost=costs.at["coal", "marginal_cost"],
            efficiency=costs.at["coal", "efficiency"], overwrite=True)

      n.add("Generator", "lignite_S", bus="DEU_S_elec", carrier="lignite", p_nom_extendable=True,
            p_nom_max=limits["lignite"], p_min_pu=0.4,
            capital_cost=costs.at["lignite", "capital_cost"], marginal_cost=costs.at["lignite", "marginal_cost"],
            efficiency=costs.at["lignite", "efficiency"], overwrite=True)

      n.add("Generator", "biomass_S", bus="DEU_S_elec", carrier="biomass CHP", p_nom_extendable=True,
            p_nom_max=limits["biomass CHP"], p_min_pu=0.33,
            capital_cost=costs.at["biomass CHP", "capital_cost"], marginal_cost=costs.at["biomass CHP", "marginal_cost"],
            efficiency=costs.at["biomass CHP", "efficiency"], overwrite=True)

      n.add("Generator", "OCGT_S", bus="DEU_S_elec", carrier="OCGT", p_nom_extendable=True,
            p_nom_max=limits["OCGT"], p_min_pu=0.2,
            capital_cost=costs.at["OCGT", "capital_cost"], marginal_cost=costs.at["OCGT", "marginal_cost"],
            efficiency=costs.at["OCGT", "efficiency"], overwrite=True)

      n.add("Generator", "ror_S", bus="DEU_S_elec", carrier="ror", p_nom_extendable=True,
            p_nom_max=limits["ror"], p_max_pu=CF_ror,
            capital_cost=costs.at["ror", "capital_cost"], marginal_cost=costs.at["ror", "marginal_cost"],
            efficiency=costs.at["ror", "efficiency"], overwrite=True)

      # Battery storage
      n.add("Bus", "battery_bus_S", carrier="battery storage")
      n.add("Store", "battery_S", bus="battery_bus_S", e_nom_extendable=True, e_cyclic=True,
            capital_cost=costs.at["battery storage", "capital_cost"])

      n.add("Link", "charge_battery_S", bus0="DEU_S_elec", bus1="battery_bus_S", p_nom_extendable=True,
            efficiency=costs.at["battery inverter", "efficiency"],
            capital_cost=costs.at["battery inverter", "capital_cost"])

      n.add("Link", "discharge_battery_S", bus0="battery_bus_S", bus1="DEU_S_elec", p_nom_extendable=True,
            efficiency=costs.at["battery inverter", "efficiency"],
            capital_cost=costs.at["battery inverter", "capital_cost"])
      
      co2_emissions_2040 = 0.12*366e6
      n.add("GlobalConstraint", "CO2Limit",
            carrier_attribute="co2_emissions",
            sense="<=",
            constant=co2_emissions_2040)
      print("co2 allowance: ", co2_emissions_2040/10e6, "tCo2")

      return n

### South

### Line

In [84]:
# Transmission line between North and South
n = build_network(costs)

line_length_km = 500  # estimated distance
line_cost_per_MW_km = costs.at["HVAC overhead", "capital_cost"]
total_line_cost = line_length_km * line_cost_per_MW_km

n.add("Line",
    name="DEU_N_to_S",
    bus0="DEU_N_elec",
    bus1="DEU_S_elec",
    s_nom=0,  # start at 0
    s_nom_extendable=True,
    capital_cost=total_line_cost,
    x=1,  
    r=1   
)


# Solve
n.optimize(solver_name="gurobi")

/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily["Inflow [MW]"] = df_daily["Inflow [GWh]"] * 1000 / 24  # MW average per hour
Index(['charge_battery_N', 'charge_battery_S'], dtype='object', name='Link')
Index(['DEU_N_elec', 'DEU_S_elec'], dtype='object', name='Bus')
Index(['DEU_N_to_S'], dtype='object', name='Line')


co2 allowance:  4.392 tCo2


INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 9/9 [00:00<00:00, 31.79it/s]
INFO:linopy.io: Writing time: 2.17s


Set parameter Username


INFO:gurobipy:Set parameter Username


...

Solved in 21091 iterations and 63.56 seconds (28.11 work units)


INFO:gurobipy:Solved in 21091 iterations and 63.56 seconds (28.11 work units)


Optimal objective  6.148055127e+10


INFO:gurobipy:Optimal objective  6.148055127e+10
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 210262 primals, 438034 duals
Objective: 6.15e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Line-ext-s-lower, Line-ext-s-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, Store-energy_balance were not assigned to the network.


('ok', 'optimal')

In [81]:
def summarize_network_results(n):
    # Total system cost
    total_cost = n.objective / 1e9  # in billion EUR

    # Generation mix per zone (energy in TWh)
    energy = n.generators_t.p.multiply(n.snapshot_weightings.generators, axis=0).sum() / 1e6  # MWh → TWh
    energy_per_zone = {}

    for gen in n.generators.index:
        zone = "North" if "_N" in gen else "South" if "_S" in gen else "Unknown"
        carrier = n.generators.loc[gen, "carrier"]
        energy_per_zone.setdefault(zone, {}).setdefault(carrier, 0)
        energy_per_zone[zone][carrier] += energy[gen]

    energy_df = pd.DataFrame(energy_per_zone).fillna(0).round(2)

    # Transmission line capacity (assumes a single line "DEU_N_to_S")
    line_capacity = round(float(n.lines.loc["DEU_N_to_S"]["s_nom_opt"]),2)

    return total_cost, energy_df, line_capacity

# Example usage
total_cost_billion_eur, generation_mix_df, line_capacity_mw = summarize_network_results(n)
print(f"Total system cost: {total_cost_billion_eur:.2f} billion EUR")
print(f"Transmission line capacity (North→South): {line_capacity_mw:.0f} MW")
print("\nGeneration mix per zone (TWh):")
print(generation_mix_df)

Total system cost: 61.48 billion EUR
Transmission line capacity (North→South): 24813 MW

Generation mix per zone (TWh):
             North   South
onwind       93.40    0.00
offwind      63.47    0.00
solar         0.00  257.57
coal          0.00    0.00
lignite       0.00    0.00
biomass CHP  19.37   19.79
OCGT         11.12   34.26
ror           8.22    8.22


In [85]:
def run_scenario(costs, line_capacity=None, optimize_line=True):
    n = build_network(costs)
    
    # Define line parameters
    line_length_km = 500
    line_cost_per_MW_km = costs.at["HVAC overhead", "capital_cost"]
    total_line_cost = line_length_km * line_cost_per_MW_km

    # Add transmission line
    n.add("Line",
          name="DEU_N_to_S",
          bus0="DEU_N_elec",
          bus1="DEU_S_elec",
          s_nom=0 if optimize_line else line_capacity,
          s_nom_extendable=optimize_line,
          capital_cost=total_line_cost,
          x=1,
          r=1)

    # Solve the network
    n.optimize(solver_name="gurobi")

    # Summarize results
    return summarize_network_results(n)

In [86]:
# Load technology data
costs = load_technology_data()

# Scenario 1: No Transmission
cost_no_line, mix_no_line, cap_no_line = run_scenario(costs, line_capacity=0, optimize_line=False)

# Scenario 2: Planned HVDC Capacity (12 GW)
cost_12GW, mix_12GW, cap_12GW = run_scenario(costs, line_capacity=12000, optimize_line=False)

# Scenario 3: Optimized Capacity
cost_opt, mix_opt, cap_opt = run_scenario(costs, optimize_line=True)

/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily["Inflow [MW]"] = df_daily["Inflow [GWh]"] * 1000 / 24  # MW average per hour
Index(['charge_battery_N', 'charge_battery_S'], dtype='object', name='Link')
Index(['DEU_N_elec', 'DEU_S_elec'], dtype='object', name='Bus')
Index(['DEU_N_to_S'], dtype='object', name='Line')


co2 allowance:  4.392 tCo2


INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 8/8 [00:00<00:00, 29.54it/s]
INFO:linopy.io: Writing time: 2.16s


Set parameter Username


INFO:gurobipy:Set parameter Username


...

INFO:gurobipy:


Solved in 21091 iterations and 62.26 seconds (28.11 work units)


INFO:gurobipy:Solved in 21091 iterations and 62.26 seconds (28.11 work units)


Optimal objective  6.148055127e+10


INFO:gurobipy:Optimal objective  6.148055127e+10
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 210262 primals, 438034 duals
Objective: 6.15e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Line-ext-s-lower, Line-ext-s-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, Store-energy_balance were not assigned to the network.


In [87]:
# Print total system costs
print("Total System Cost (billion EUR):")
print(f"No Line: {cost_no_line:.2f}")
print(f"12 GW Line: {cost_12GW:.2f}")
print(f"Optimized Line: {cost_opt:.2f}\n")

# Print transmission line capacities
print("Transmission Line Capacity (MW):")
print(f"No Line: {cap_no_line}")
print(f"12 GW Line: {cap_12GW}")
print(f"Optimized Line: {cap_opt}\n")

# Print generation mix per zone
print("Generation Mix per Zone (TWh):")
print("No Line:")
print(mix_no_line)
print("\n12 GW Line:")
print(mix_12GW)
print("\nOptimized Line:")
print(mix_opt)

Total System Cost (billion EUR):
No Line: 81.95
12 GW Line: 62.58
Optimized Line: 61.48

Transmission Line Capacity (MW):
No Line: 0.0
12 GW Line: 12000.0
Optimized Line: 24813.29

Generation Mix per Zone (TWh):
No Line:
             North   South
onwind       22.08   12.42
offwind      48.79    0.00
solar        99.70  236.70
coal          0.00    0.00
lignite       0.00    0.00
biomass CHP  16.90   21.19
OCGT         10.24   35.14
ror           8.22    8.22

12 GW Line:
             North   South
onwind       75.48   15.57
offwind      65.95    0.00
solar         0.00  256.24
coal          0.00    0.00
lignite       0.00    0.00
biomass CHP  19.73   21.90
OCGT         10.43   34.95
ror           8.22    8.22

Optimized Line:
             North   South
onwind       93.40    0.00
offwind      63.47    0.00
solar         0.00  257.57
coal          0.00    0.00
lignite       0.00    0.00
biomass CHP  19.37   19.79
OCGT         11.12   34.26
ror           8.22    8.22
